# Capstone — Análise Exploratória de Dados (EDA)
## Previsão de Vendas no Varejo com Prophet

**Dataset:** Rossmann Store Sales — Kaggle  
**Autor:** Raphael Costa de Oliveira Bomeisel  
**Programa:** PADS — Insper · 2026

---

### O que é e para que serve este notebook?

Antes de treinar qualquer modelo, precisamos **entender os dados**. A Análise Exploratória (EDA) 
cumpre três funções essenciais no projeto:

1. **Detectar problemas** — valores ausentes, dias com loja fechada, inconsistências
2. **Revelar padrões** — sazonalidade semanal, anual, efeito de promoções
3. **Guiar decisões de modelagem** — quais variáveis incluir, qual modo de sazonalidade usar

Cada seção responde a uma pergunta de negócio específica e documenta o que aprendemos 
e como isso moldou o modelo final.

---

### Seções
1. Carregamento e visão geral dos dados
2. Limpeza e pré-processamento
3. Distribuição das vendas
4. Sazonalidade semanal e anual
5. Impacto de promoções
6. Série temporal da Loja 1 (foco do modelo)
7. Correlações entre variáveis
8. Conclusões e implicações para o modelo

---
### Setup

Carregamos as bibliotecas de análise e visualização, além dos módulos do próprio projeto 
(`data_loader` e `preprocessing`). Usar os mesmos módulos do pipeline garante que a EDA 
reflita exatamente os dados que o modelo vai ver — sem inconsistências entre o notebook 
e o código de produção.

In [ ]:
import sys
from pathlib import Path

# FIX #7: use resolve() so the path works regardless of CWD
sys.path.insert(0, str(Path().resolve().parent / 'src'))

# FIX #3: ensure output directory exists before any savefig call
FIGURES_DIR = Path('..') / 'reports' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# FIX #2: guard the custom-module imports with a clear error message
try:
    from data_loader import load_processed
    from preprocessing import clean, get_store_series
except ImportError as e:
    raise ImportError(
        f"Módulos do projeto não encontrados: {e}\n"
        "Verifique se ../src/data_loader.py e ../src/preprocessing.py existem "
        "e se o notebook está sendo executado a partir do diretório notebooks/."
    ) from e

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.titlesize'] = 14

print('✅ Imports OK')
print(f'📁 Figuras serão salvas em: {FIGURES_DIR.resolve()}')

---
## 1. Carregamento e visão geral

**O que queremos saber:**  
Como o dataset está estruturado? Quantas linhas e colunas existem? Quais variáveis estão 
disponíveis e quais têm valores ausentes?

O dataset Rossmann é composto por dois arquivos do Kaggle:
- `train.csv` — histórico de vendas diárias por loja (Jan 2013 – Jul 2015)
- `store.csv` — características de cada loja (tipo, sortimento, concorrência)

O módulo `data_loader` já fez o merge dos dois arquivos. Aqui vamos inspecionar o resultado.

In [ ]:
df_raw = load_processed('rossmann_merged.csv')
print(f'Shape bruto: {df_raw.shape}')
df_raw.head()

In [ ]:
print('=== Tipos de dados ===')
print(df_raw.dtypes)
print()
print('=== Valores ausentes ===')
print(df_raw.isnull().sum()[df_raw.isnull().sum() > 0])

**O que os valores ausentes nos dizem:**  

As colunas com NaN são `CompetitionDistance`, `CompetitionOpenSinceMonth/Year`, `Promo2SinceWeek/Year` e `PromoInterval`. 
Isso **não é um erro** — significa que algumas lojas simplesmente não têm concorrente próximo registrado 
ou não participam da Promo2. A estratégia correta é preencher com a mediana (variáveis numéricas) 
ou com 0/"None" (variáveis categóricas), que é exatamente o que a função `clean()` faz.

**Implicação para o modelo:** `CompetitionDistance` é uma feature potencialmente relevante 
(lojas com concorrentes próximos podem ter vendas menores), mas como há NaNs, precisamos 
imputar antes de usar. Mantivemos ela no pipeline de limpeza.

In [ ]:
# FIX #4: wrap describe() in display() so it renders even with preceding print()
from IPython.display import display
print('=== Estatísticas descritivas (Sales e Customers) ===')
display(df_raw[['Sales', 'Customers']].describe())

**O que descobrimos:**  

- A mediana de Sales é maior que zero, mas o mínimo é 0 — isso indica dias em que a loja estava fechada.  
- A amplitude de vendas é enorme (mín 0, máx ~41.000 €) — reflete a diversidade entre os 1.115 tipos de loja.
- O mesmo padrão aparece em Customers: min 0 (fechada), max ~7.388.  

**Implicação:** antes de modelar precisamos **remover os dias com Sales = 0 ou Open = 0**, 
pois esses zeros não representam uma queda real de vendas — representam ausência de operação. 
Incluí-los enviesaria qualquer modelo de série temporal.

---
## 2. Limpeza dos dados

**O que queremos:** uma série consistente, sem dias fechados, sem NaNs e com features de 
data derivadas (ano, mês, dia da semana) que o modelo e a EDA vão usar.

A função `clean()` aplica o pipeline completo:
- Remove dias com `Open == 0` ou `Sales == 0`
- Preenche NaNs nas variáveis de concorrência e promoção
- Mapeia `StateHoliday` para texto legível
- Cria features de data (`Year`, `Month`, `DayOfMonth`, `WeekOfYear`, `IsWeekend`)
- Cria flag `CompetitionOpen` (concorrente já estava aberto naquela data?)
- Calcula `LogSales` (log-transformação das vendas — útil para análises que assumem normalidade)

In [ ]:
df = clean(df_raw)
print(f'Shape limpo: {df.shape}')
print(f'Período: {df.Date.min().date()} → {df.Date.max().date()}')
print(f'Lojas únicas: {df.Store.nunique()}')
print(f'Linhas removidas: {len(df_raw) - len(df):,} ({(1 - len(df)/len(df_raw))*100:.1f}% do total)')

**O que descobrimos:**  

Cerca de 15-17% das linhas são removidas — todos os domingos em que as lojas estavam fechadas 
e alguns feriados. Esse percentual é esperado (lojas alemãs têm restrições de abertura aos domingos).

**Contribuição para o projeto:** o dataset limpo é o que o Prophet vai treinar. 
Como o Prophet foi projetado para séries com gaps (dias sem dados), não precisamos 
preencher os domingos com zeros — simplesmente os omitimos e o modelo lida com isso nativamente.

---
## 3. Distribuição das Vendas

**O que queremos saber:**  
Como as vendas se distribuem? A distribuição é simétrica? Existem outliers? 
Há diferenças relevantes entre os tipos de loja?

Entender a distribuição das vendas é crucial por dois motivos:
1. **Escolha da métrica:** se as vendas forem muito assimétricas, o MAPE pode ser instável.
2. **Escolha do modo de sazonalidade:** distribuições mais homogêneas favorecem o modo aditivo;
   distribuições com alta variância relativa podem favorecer o multiplicativo.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histograma
axes[0].hist(df['Sales'], bins=80, color='steelblue', edgecolor='white')
axes[0].set_title('Distribuição das Vendas Diárias')
axes[0].set_xlabel('Vendas (€)')
axes[0].set_ylabel('Frequência')
axes[0].axvline(df['Sales'].median(), color='navy', linestyle='--', linewidth=1.5, label=f'Mediana: €{df["Sales"].median():,.0f}')
axes[0].axvline(df['Sales'].mean(),   color='red',  linestyle='--', linewidth=1.5, label=f'Média: €{df["Sales"].mean():,.0f}')
axes[0].legend(fontsize=9)

# Boxplot por tipo de loja
df.boxplot(column='Sales', by='StoreType', ax=axes[1], patch_artist=True)
axes[1].set_title('Vendas por Tipo de Loja')
axes[1].set_xlabel('Tipo de Loja')
axes[1].set_ylabel('Vendas (€)')
plt.suptitle('')  # suprime o supertítulo automático gerado pelo boxplot
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'distribuicao_vendas.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Assimetria (skewness): {df["Sales"].skew():.2f}')
print(f'Curtose: {df["Sales"].kurt():.2f}')

**O que descobrimos:**  

- A distribuição é **assimétrica à direita** (cauda longa) — a maioria das lojas vende entre €3.000-€8.000/dia, mas algumas chegam a €40.000+ (lojas tipo b).
- **Lojas tipo b** têm mediana muito superior às demais — são as lojas de maior porte.
- A média > mediana confirma a assimetria. Um modelo que erra €500 em lojas que vendem €4.000 erra ~12%; o mesmo erro em lojas tipo b que vendem €15.000 erra apenas ~3%.

**Contribuição para o projeto:**  
- Isso justifica o uso do **MAPE como métrica principal** — ele normaliza o erro pela escala de cada loja, tornando os resultados comparáveis.
- A assimetria indica que o modo **aditivo** é mais adequado para a Loja 1 (histórico estável, sem crescimento explosivo), porque a sazonalidade em € é aproximadamente constante — não proporcional ao nível de vendas. Isso foi **confirmado empiricamente** no Experimento 1.

---
## 4. Sazonalidade — Semanal e Mensal

**O que queremos saber:**  
Existe um padrão sistemático de vendas por dia da semana? E por mês do ano? 
Qual é a amplitude desses padrões?

Esta é **a análise mais crítica para o modelo**. O Prophet usa séries de Fourier para 
capturar sazonalidade — mas só vai funcionar bem se a sazonalidade for de fato periódica 
e estável ao longo do tempo. Aqui verificamos se isso é verdade.

> **Amplitude semanal** = diferença entre o dia de maior e menor venda média.  
> Se for grande e consistente, é um sinal forte que o Prophet deve aprender.

> **Nota sobre codificação:** No dataset Rossmann, `DayOfWeek` usa codificação 1–7 
> (1 = Segunda, 7 = Domingo), conforme o Kaggle. Esta seção usa essa coluna diretamente. 
> Na Seção 6b usamos `dt.dayofweek` (0–6) derivado do datetime — ambos são consistentes 
> entre si após o mapeamento.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# FIX #5: DayOfWeek no dataset Rossmann é 1-7 (1=Segunda, 7=Domingo)
dow_map = {1:'Seg', 2:'Ter', 3:'Qua', 4:'Qui', 5:'Sex', 6:'Sáb', 7:'Dom'}
dow_avg = df.groupby('DayOfWeek')['Sales'].mean()
dow_avg.index = dow_avg.index.map(dow_map)
dow_avg.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Vendas Médias por Dia da Semana (todas as lojas)')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=0)
axes[0].set_ylabel('Vendas Médias (€)')

# Por mês
month_map = {1:'Jan', 2:'Fev', 3:'Mar', 4:'Abr', 5:'Mai', 6:'Jun',
             7:'Jul', 8:'Ago', 9:'Set', 10:'Out', 11:'Nov', 12:'Dez'}
month_avg = df.groupby('Month')['Sales'].mean()
month_avg.index = month_avg.index.map(month_map)
month_avg.plot(kind='bar', ax=axes[1], color='seagreen', edgecolor='white')
axes[1].set_title('Vendas Médias por Mês (todas as lojas)')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=30)
axes[1].set_ylabel('Vendas Médias (€)')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'sazonalidade.png', dpi=150, bbox_inches='tight')
plt.show()

# Amplitude (usando a coluna original DayOfWeek 1-7)
raw_dow = df.groupby('DayOfWeek')['Sales'].mean()
amp = raw_dow.max() - raw_dow.min()
print(f'Amplitude semanal média (todas as lojas): €{amp:,.0f}')
print(f'Dia com MAIOR venda média: DayOfWeek={raw_dow.idxmax()} — {dow_map[raw_dow.idxmax()]}')
print(f'Dia com MENOR venda média: DayOfWeek={raw_dow.idxmin()} — {dow_map[raw_dow.idxmin()]}')

**O que descobrimos:**

**Sazonalidade semanal:**
- Há um padrão claro e consistente. **Segunda** e quinta tendem a liderar as vendas; domingo tem queda abrupta (restrições legais de abertura na Alemanha). A amplitude (diferença pico-vale) é da ordem de €700-800 — grande o suficiente para ter impacto real nas previsões.
- Esse padrão é o que o Prophet captura com a sazonalidade semanal (Fourier, período P=7).

**Sazonalidade anual:**
- Pico em dezembro (Natal) e queda em janeiro-fevereiro. O mês de abril tem uma pequena recuperação (Páscoa). Julho-agosto mostram leve queda (férias de verão alemãs).
- Esse padrão é capturado pelo Prophet com a sazonalidade anual (Fourier, período P=365.25).

**Contribuição para o projeto:**  
Ambas as sazonalidades são **estáveis ao longo dos anos** (não crescem proporcionalmente com a tendência), o que confirma que o **modo aditivo** é o correto. No modo multiplicativo, a amplitude cresceria junto com a tendência — e isso **não** é o que observamos nos dados. Esse foi o diagnóstico que levou à decisão do Experimento 1.

---
## 5. Impacto de Promoções

**O que queremos saber:**  
A promoção (`Promo=1`) realmente aumenta as vendas, ou é apenas ruído? Em quanto?
O efeito é consistente entre lojas ou varia muito?

Esta análise motivou **a decisão mais importante do projeto**: adicionar a promoção 
como **regressor externo** no Prophet (`model.add_regressor("Promo")`). Para justificar 
essa escolha para a banca, precisamos mostrar evidência estatística do efeito.

> **O que é um regressor externo no Prophet?**  
> É uma variável que o modelo usa como entrada adicional além da própria série temporal. 
> O Prophet aprende o coeficiente β (uplift médio) a partir do histórico e o aplica nos dias futuros.

In [ ]:
promo_avg = df.groupby('Promo')['Sales'].agg(['mean', 'median', 'std', 'count']).round(0)
promo_avg.index = ['Sem Promoção', 'Com Promoção']
promo_avg.columns = ['Média (€)', 'Mediana (€)', 'Desvio Padrão (€)', 'Observações']
print('=== Vendas: Com vs Sem Promoção ===')
print(promo_avg.to_string())

uplift_media   = (promo_avg.loc['Com Promoção','Média (€)']   / promo_avg.loc['Sem Promoção','Média (€)']   - 1) * 100
uplift_mediana = (promo_avg.loc['Com Promoção','Mediana (€)'] / promo_avg.loc['Sem Promoção','Mediana (€)'] - 1) * 100
print(f'\n📈 Uplift na MÉDIA   com promoção: +{uplift_media:.1f}%')
print(f'📈 Uplift na MEDIANA com promoção: +{uplift_mediana:.1f}%')

# Boxplot comparativo
fig, ax = plt.subplots(figsize=(8, 4))
df.boxplot(column='Sales', by='Promo', ax=ax, patch_artist=True)
ax.set_xticklabels(['Sem Promoção (0)', 'Com Promoção (1)'])
ax.set_title('Distribuição de Vendas — Com vs Sem Promoção')
ax.set_xlabel('')
ax.set_ylabel('Vendas (€)')
plt.suptitle('')  # suprime o supertítulo automático gerado pelo boxplot
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'impacto_promocao.png', dpi=150, bbox_inches='tight')
plt.show()

**O que descobrimos:**

- Dias com promoção têm vendas médias ~17-25% maiores que dias sem promoção.
- O efeito aparece tanto na média quanto na mediana — ou seja, não é puxado por outliers.
- O desvio padrão é similar nos dois grupos — a variabilidade interna não aumenta com a promoção.

**Por que a sazonalidade sozinha não captura isso:**  
A sazonalidade semanal diz segunda vende mais que terça em média. Mas ela não sabe se 
aquela segunda específica tem promoção. Um dia de promoção em qualquer dia da semana 
tem um uplift **adicional** — e é exatamente esse uplift que o regressor `Promo` aprende.

**Resultado no modelo:**  
Adicionar `Promo` como regressor reduziu o MAPE de **15,47% → 7,54%** — uma queda de 51%. 
Isso é a evidência empírica mais forte do projeto: **a variável mais importante não é um 
parâmetro do Prophet, mas uma informação de negócio que trouxemos de fora**.

---
## 6. Série Temporal — Loja 1

**O que queremos saber:**  
Como a série da loja que vamos modelar se comporta ao longo do tempo? 
Existe tendência? A sazonalidade é estável? Há mudanças estruturais visíveis?

Escolhemos a **Loja 1** como caso de referência porque:
- É do tipo `a` — a mais comum no dataset (602 de 1115 lojas)
- Tem histórico completo e sem gaps longos
- Tem sortimento básico, representativo do caso médio

A **média móvel de 28 dias** (4 semanas) suaviza a variação diária e revela 
a tendência de médio prazo sem o ruído semanal.

In [ ]:
STORE_ID = 1
series = get_store_series(df, store_id=STORE_ID)
series = series.sort_values('ds').copy()
series['ma_28'] = series['y'].rolling(28).mean()

fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(series['ds'], series['y'],     alpha=0.35, color='steelblue', linewidth=0.7, label='Vendas diárias')
ax.plot(series['ds'], series['ma_28'], color='navy',  linewidth=2.2, label='Média móvel 28 dias')
ax.set_title(f'Série Temporal de Vendas — Loja {STORE_ID}')
ax.set_xlabel('Data')
ax.set_ylabel('Vendas (€)')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'serie_temporal_loja1.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Observações (dias aberta): {len(series)}')
print(f'Período: {series.ds.min().date()} → {series.ds.max().date()}')
print(f'Vendas médias: €{series.y.mean():,.0f}/dia')
print(f'Vendas mínimas: €{series.y.min():,.0f}  |  máximas: €{series.y.max():,.0f}')
print(f'Amplitude diária: €{series.y.max() - series.y.min():,.0f}')

**O que descobrimos:**

- A série tem **~900 observações** (dias abertos) ao longo de ~2,5 anos.
- A **tendência** é relativamente estável — não há crescimento ou queda expressiva. Isso 
  justifica usar um `changepoint_prior_scale` baixo (0.05): não precisamos de uma tendência 
  muito flexível, pois a loja não está mudando de patamar.
- A **oscilação semanal** é visível como ruído de alta frequência — esse é exatamente 
  o padrão que a sazonalidade de Fourier (período 7) deve capturar.
- Os **picos de fim de ano** (dezembro) são claros na série suavizada — capturados pela 
  sazonalidade anual (período 365.25).

**Contribuição para o projeto:**  
A estabilidade da tendência e a regularidade das sazonalidades confirmam que o Prophet 
é uma boa escolha para esta série. Se a tendência fosse explosiva ou as sazonalidades 
mudassem de formato ao longo do tempo, precisaríamos de modelos mais complexos.

### 6b. Amplitude semanal da Loja 1

Vamos calcular a **amplitude semanal real** da Loja 1 — esse número vai aparecer diretamente 
na análise do Experimento 1, onde comparamos o que o modelo reproduz vs. o que os dados mostram.

> **Nota:** aqui derivamos o dia da semana diretamente do campo datetime (`ds`) com 
> `dt.dayofweek`, que retorna 0=Segunda … 6=Domingo. Isso é diferente da coluna `DayOfWeek` 
> original do Rossmann (1=Segunda … 7=Domingo usada na Seção 4), mas ambas representam 
> o mesmo padrão — só o mapeamento de labels muda.

In [ ]:
# FIX #5: dt.dayofweek retorna 0-6 (0=Seg, 6=Dom) — mapeamento consistente com essa escala
series['dow'] = series['ds'].dt.dayofweek
dow_labels = {0:'Seg', 1:'Ter', 2:'Qua', 3:'Qui', 4:'Sex', 5:'Sáb', 6:'Dom'}
dow_loja1 = series.groupby('dow')['y'].mean()
amp_real = dow_loja1.max() - dow_loja1.min()

fig, ax = plt.subplots(figsize=(8, 4))
dow_loja1.rename(index=dow_labels).plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title(f'Loja {STORE_ID} — Vendas Médias por Dia da Semana')
ax.set_xlabel('')
ax.set_ylabel('Vendas Médias (€)')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

print(f'Amplitude semanal REAL — Loja {STORE_ID}: €{amp_real:,.0f}')
print(f'Dia de PICO : {dow_labels[dow_loja1.idxmax()]} (€{dow_loja1.max():,.0f})')
print(f'Dia de VALE : {dow_labels[dow_loja1.idxmin()]} (€{dow_loja1.min():,.0f})')
print()
print('▶ Esse número (~720 €) será o benchmark do Experimento 1:')
print('  modo multiplicativo → amplitude 0 €  (modelo plano)')
print('  modo aditivo        → amplitude ~698 € (recuperou o padrão)')

---
## 7. Correlações entre variáveis

**O que queremos saber:**  
Quais variáveis numéricas se correlacionam com as vendas? Alguma poderia ser incluída 
como regressor adicional no Prophet? Existe multicolinearidade entre regressores potenciais?

A matriz de correlação é um diagnóstico rápido — correlação alta com Sales indica uma 
variável potencialmente útil; correlação alta entre variáveis independentes indica 
multicolinearidade (problema para modelos lineares, menos crítico para o Prophet).

> **Atenção:** correlação mede relação linear. Efeitos não-lineares (ex.: sazonalidade) 
> não aparecem aqui. Por isso a EDA não substitui os experimentos — ela complementa.

In [ ]:
# FIX #6: verificar que todas as colunas existem antes de calcular correlação
num_cols = ['Sales', 'Customers', 'CompetitionDistance', 'Promo', 'Promo2', 'CompetitionOpen']
missing_cols = [c for c in num_cols if c not in df.columns]
if missing_cols:
    print(f'⚠️  Colunas ausentes no DataFrame: {missing_cols}')
    print('   Verifique se clean() gerou todas as features esperadas.')
    num_cols = [c for c in num_cols if c in df.columns]

corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            square=True, ax=ax, linewidths=0.5)
ax.set_title('Matriz de Correlação — Variáveis Numéricas')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'correlacao.png', dpi=150, bbox_inches='tight')
plt.show()

print('Correlação de cada variável com Sales:')
print(corr['Sales'].sort_values(ascending=False).to_string())

**O que descobrimos:**

| Variável | Correlação com Sales | Interpretação |
|----------|---------------------|---------------|
| `Customers` | ~0.82 (alta) | Mais clientes = mais vendas. Causal óbvio. |
| `Promo` | ~0.35 (moderada) | Promoção aumenta vendas — evidência de que é um regressor útil. |
| `CompetitionDistance` | ~-0.10 (fraca) | Concorrente mais próximo = ligeira queda nas vendas. Fraco. |
| `Promo2` | ~0.02 (negligível) | Promoção recorrente tem efeito mínimo no total. |
| `CompetitionOpen` | ~-0.05 (negligível) | Se o concorrente está aberto, impacto muito pequeno. |

**Por que não usamos `Customers` como regressor:**  
`Customers` tem alta correlação com `Sales` — mas é uma variável que **não conhecemos no futuro**. 
Não sabemos quantos clientes vão entrar na loja amanhã. Incluí-la seria data leakage.

**Por que usamos `Promo`:**  
O calendário de promoções é **decidido com antecedência** pelo gestor de loja — 
é uma variável que genuinamente podemos conhecer no momento da previsão. 
Isso a torna um regressor válido e poderoso.

**Contribuição para o projeto:**  
A análise de correlação confirmou que `Promo` é o regressor externo mais promissor. 
Os demais (CompetitionDistance, Promo2) têm correlação baixa e foram excluídos do modelo 
final — mais uma decisão guiada por evidência, não por suposição.

---
## 8. Conclusões da EDA e Implicações para o Modelo

A EDA não é apenas exploração — cada achado se traduz em uma **decisão de modelagem**.

| Achado da EDA | Decisão no modelo |
|---------------|------------------|
| Sazonalidade semanal estável (amplitude ~€720, constante ao longo dos anos) | `seasonality_mode="additive"` |
| Sazonalidade anual clara (pico Natal, queda Jan) | `yearly_seasonality=True` |
| Tendência plana (sem crescimento expressivo) | `changepoint_prior_scale=0.05` (padrão baixo) |
| Promoção aumenta vendas em ~20-25% | `model.add_regressor("Promo")` |
| `Customers` não disponível no futuro | Excluído como regressor |
| `CompetitionDistance` e `Promo2` têm correlação < 0.1 | Excluídos do modelo final |
| Distribuição assimétrica entre lojas | MAPE como métrica principal (normaliza por escala) |
| Domingos ausentes (lojas fechadas) | Nenhuma imputação necessária — Prophet tolera gaps |

**Resultado esperado do modelo:**  
Com essas decisões, o modelo atingiu MAPE de **8,84%** no cenário realista (promoção futura 
estimada pelo padrão histórico) e **7,54%** no cenário ótimo (promoção futura conhecida). 
Ambos abaixo da meta de <10%.

---

**Próximo passo:** [`src/model.py`](../src/model.py) — treinar o Prophet com os parâmetros 
justificados por esta EDA e validar com hold-out temporal (últimos 42 dias).

**Experimentos documentados:** [`experimentos/`](../experimentos/) — testes sistemáticos 
de modo aditivo vs multiplicativo e impacto do regressor Promo.